# Notebook 00: Generación del mapping cultivo–distrito (v2)

**Objetivo:** generar de forma reproducible la tabla `(región, cultivo) → piso ecológico + distrito NASA` usada en el merge del notebook 03.

| Campo | Valor |
|-------|-------|
| Input producción | `OUTPUTS/midagri_largo.csv` (notebook 01) |
| Input referencia | `BDS/mapping/mapping_cultivo_distrito.csv` (v1, solo comparación) |
| Inventario distritos | 14 puntos NASA (mismo catálogo que notebook 02) |
| Output completo | `BDS/mapping/mapping_cultivo_distrito_v2.csv` |
| Output pipeline | `BDS/mapping/mapping_cultivo_distrito_v2_pipeline.csv` |
| Reporte | `DOCUMENTACIÓN/reporte_mapping_v1_vs_v2.md` |

**Orden del pipeline:** 01 → **00** → 02 → 03 → … (00 corre tras 01; no depende de la descarga NASA).

---

## Por qué existe este mapping: el problema que resuelve

MIDAGRI publica producción agrícola **por región** (departamento): una fila dice, por ejemplo, «Ica produjo X toneladas de espárrago en marzo», pero **no dice en qué valle, distrito ni altitud** se cosechó. En cambio, los datos climáticos de NASA POWER son **puntuales**: cada observación corresponde a un par `(latitud, longitud)` concreto, en nuestro caso a **14 distritos** elegidos como representativos de la diversidad agroclimática del Perú.

Sin una tabla puente, no se puede cruzar producción con clima de forma honesta. El mapping responde a la pregunta:

> *«Si MIDAGRI dice que hay producción de cultivo C en la región R, ¿qué serie climática mensual deberíamos asociarle?»*

La respuesta no puede ser perfecta: es una **aproximación documentada**, no una georreferenciación parcelaria. Por eso v2 incorpora nivel de confianza, justificación por fila y exclusiones explícitas en lugar de inventar distritos donde la agronomía no cuadra.

---

## Criterios de diseño: cómo se pensó la asignación

### 1. Unidad espacial: el «piso ecológico» como puente

No asignamos cultivo → distrito directamente. Primero definimos **en qué tipo de ambiente agroclimático** tiene sentido que crezca el cultivo (costa, yunga, sierra, selva alta, puna, etc.), y luego buscamos **si esa región tiene un distrito NASA que represente ese piso**.

El piso ecológico condensa altitud, régimen térmico, humedad y patrones de uso del suelo sin necesitar polígonos GIS. En el proyecto usamos estos pisos (no exhaustivos del Perú, pero suficientes para los 14 puntos):

| Piso | Distrito representativo | Región |
|------|-------------------------|--------|
| costa | Chincha Alta | Ica |
| costa | Virú | La Libertad |
| sierra | Huamachuco | La Libertad |
| yunga | Cascas | La Libertad |
| bosque_seco | Tambogrande | Piura |
| valle_chira | Sullana | Piura |
| sierra | Canchaque | Piura |
| selva_alto_mayo | Moyobamba | San Martín |
| selva_huallaga | Tocache | San Martín |
| selva_alta | Perené | Junín |
| selva_baja | Río Tambo | Junín |
| sierra | El Tambo | Junín |
| altiplano_lacustre | Ilave | Puno |
| puna_alta | Ayaviri | Puno |

**Criterio:** cada región del estudio tiene entre 1 y 3 distritos, cubriendo sus principales ambientes productivos. No intentamos mapear los 1.800+ distritos del Perú; priorizamos **representatividad ecológica** sobre resolución administrativa.

### 2. Taxonomía agronómica por cultivo (`CULTIVO_APTITUD`)

Para cada uno de los ~54 cultivos MIDAGRI definimos una **lista ordenada de pisos compatibles**, basada en:

- **Requerimientos térmicos y hídricos** del cultivo (p. ej. quinua y papa → puna/altiplano/sierra; espárrago → costa irrigada).
- **Patrones de producción nacional** conocidos (agroexportación en costa, cacao en selva baja, cítricos en valles y yungas).
- **Categoría agronómica** auxiliar (frutales, hortalizas exportación, andinos, forrajeros…) para agrupar en reportes, no para decidir el distrito.

El **orden importa**: el primer piso de la lista es el más representativo a nivel nacional. Ejemplos:

- `esparrago` → solo `costa` (Ica es el epicentro exportador).
- `papa` → `sierra`, `puna_alta`, `altiplano_lacustre`, `yunga` (tubérculo andino; la prioridad favorece sierra cuando existe).
- `uva` → `yunga` antes que `costa` (uva de mesa en ceja de selva es agronómicamente más típica que uva costera, aunque ambas existen).

**Regla automática:** para cada par `(región, cultivo)` con producción MIDAGRI > 0, se recorre la lista de pisos en orden y se asigna el **primer piso que exista en esa región** según el inventario de distritos.

### 3. Filtro de producción real

Solo se mapean combinaciones `(región, cultivo)` cuya **suma de producción mensual 2020–2025 sea estrictamente > 0**. Los ceros estructurales («este cultivo no se da aquí») no generan filas en el mapping. Esto evita inflar el dataset integrado con cruces climáticos sin sustento productivo.

### 4. Overrides regionales (`OVERRIDES`)

La regla automática falla cuando **la geografía productiva real de una región difiere del primer piso nacional**. En esos casos se aplica un override manual documentado, con justificación textual. Criterios para escribir un override:

- **Concentración espacial conocida:** caña en Virú (La Libertad), cacao en Tocache (San Martín), plátano en Sullana (Piura).
- **Corrección de v1:** casos donde el mapping anterior asignaba costa cuando la producción libertense es de yunga (uva → Cascas, frijol → Cascas).
- **Heterogeneidad intra-regional:** Puno tiene Ilave (altiplano lacustre) y Ayaviri (puna alta); papa y quinua van a Ayaviri por volumen, alfalfa a Ilave por forraje lacustre.
- **Proxy débil aceptado con confianza baja:** papa en Ica se asigna a Chincha Alta (única costa disponible) aunque parte de la producción regional provenga de zonas altas — se marca `baja` confianza.

Cada override fija explícitamente `(piso, distrito, confianza, justificación)` y la regla queda registrada como `override_regional`.

### 5. Exclusiones explícitas (`EXCLUIR_EXPLICITO`)

Preferimos **no mapear** antes que forzar un distrito incorrecto. Se excluyen combos cuando:

- El volumen es marginal y agronómicamente absurdo (quinua en Ica, oca/olluco en costa).
- El cultivo no tiene piso compatible en ningún distrito de la región (cacao en Puno, cacao en La Libertad sin relevancia).
- El registro MIDAGRI probablemente es residual o error de clasificación.

Las exclusiones quedan en `mapping_v2_excluidos.csv` con motivo escrito.

### 6. Niveles de confianza

| Nivel | Cuándo se asigna |
|-------|------------------|
| **alta** | Override documentado, o primer piso de la taxonomía coincide con un distrito de la región |
| **media** | Segundo o tercer piso compatible, o override con ambigüedad moderada (uva Piura en valle Chira) |
| **baja** | Proxy forzado: la región no tiene el piso ideal y se usa el único distrito disponible (papa en Ica → costa) |

La confianza **no es una métrica estadística**; es una etiqueta de trazabilidad para quien lea o defienda el trabajo.

### 7. Algoritmo de asignación (resumen)

```
PARA cada (región, cultivo) con producción MIDAGRI > 0:
  SI está en EXCLUIR_EXPLICITO → no mapear (registrar motivo)
  SI está en OVERRIDES         → usar piso/distrito/justificación del override
  SI NO hay taxonomía para el cultivo → no mapear
  SI NO hay intersección entre pisos del cultivo y pisos de la región → no mapear
  SINO → primer piso compatible en orden de prioridad → distrito del inventario
```

### 8. Limitaciones que hay que tener presentes

1. **Un clima por (región, cultivo):** todos los meses de espárrago en Ica comparten la serie de Chincha Alta, aunque parte del espárrago pueda estar en otros valles.
2. **Producción regional ≠ distrito:** MIDAGRI agrega departamentos enteros; el clima es de un punto.
3. **Cultivos del mismo piso comparten clima idéntico** en el merge — las correlaciones por cultivo reflejan sobre todo patrones de cosecha, no sensibilidades climáticas independientes (ver guion de defensa).
4. **Sin área cosechada:** no distinguimos si un cambio de producción es por rendimiento o por superficie.
5. **Reproducibilidad vs. verdad de campo:** cambiar `CULTIVO_APTITUD` u `OVERRIDES` cambia el dataset; por eso viven en código versionado, no en Excel ad hoc.

### 9. Relación con v1 y salidas del notebook

El mapping **v1** (`mapping_cultivo_distrito.csv`) se conserva solo para comparación. La **v2** mejora trazabilidad (confianza, justificación, exclusiones) y corrige casos agronómicos mal asignados. Al ejecutar este notebook se generan:

- `mapping_cultivo_distrito_v2.csv` — tabla completa con metadatos.
- `mapping_cultivo_distrito_v2_pipeline.csv` — 4 columnas (`region`, `cultivo`, `piso_asignado`, `distrito`) consumidas por el notebook 03.
- `reporte_mapping_v1_vs_v2.md` — diff legible entre versiones.

---

### Qué hace la siguiente celda de código

Setup: rutas del proyecto e imports.

In [ ]:
# ====================================================================
# CELDA 1 — Setup: rutas del proyecto e imports
# ====================================================================
# Propósito:
#   Preparar el notebook 00 que genera el mapping cultivo→distrito (v2).
#   Resuelve la raíz del repositorio desde distintos directorios de ejecución
#   y define todas las rutas de entrada/salida (MIDAGRI, mapping v1/v2, reportes).
# Contexto en el pipeline:
#   Corre DESPUÉS del notebook 01 (necesita midagri_largo.csv) y ANTES del 03
#   (que consume mapping_cultivo_distrito_v2_pipeline.csv en el merge).
# Salida de esta celda: variables de ruta listas; impresión de ROOT en consola.
# ====================================================================

# pathlib.Path: rutas multiplataforma relativas a la raíz del repo
from pathlib import Path

# pandas: lectura de CSV de producción y escritura del mapping tabular
import pandas as pd

# --- Resolución de ROOT según desde dónde se ejecute el kernel Jupyter ---
# Si cwd es notebooks/ → subir dos niveles hasta DM_TF/
# Si cwd es SCRIPTS/  → subir un nivel
# Si no hay OUTPUTS/ en cwd pero sí en el padre → asumir subdirectorio del repo
ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent
elif not (ROOT / "OUTPUTS").exists() and (ROOT.parent / "OUTPUTS").exists():
    ROOT = ROOT.parent

# Carpetas principales del proyecto
BDS = ROOT / "BDS"                    # datos crudos y mapping generado
OUTPUTS = ROOT / "OUTPUTS"            # CSV intermedios (midagri_largo, reportes)
DOCS = ROOT / "DOCUMENTACIÓN"         # reporte markdown comparativo v1 vs v2
MAPPING_DIR = BDS / "mapping"         # destino de mapping_cultivo_distrito_v2*.csv
MAPPING_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# --- Rutas de archivos específicos del flujo mapping ---
RUTA_MIDAGRI = OUTPUTS / "midagri_largo.csv"                              # insumo: producción por región×cultivo
RUTA_V1 = MAPPING_DIR / "mapping_cultivo_distrito.csv"                    # mapping legacy de referencia
RUTA_V1_LEGACY = MAPPING_DIR / "mapping_cultivo_distrito_v1_legacy.csv"   # respaldo automático de v1
RUTA_V2 = MAPPING_DIR / "mapping_cultivo_distrito_v2.csv"                 # mapping completo con metadatos
RUTA_V2_PIPELINE = MAPPING_DIR / "mapping_cultivo_distrito_v2_pipeline.csv"  # 4 columnas para notebook 03
RUTA_COMP = OUTPUTS / "reporte_mapping_v1_vs_v2.csv"                      # tabla comparativa fila a fila
RUTA_MD = DOCS / "reporte_mapping_v1_vs_v2.md"                             # informe legible para documentación
RUTA_EXCL = OUTPUTS / "mapping_v2_excluidos.csv"                          # combos sin distrito asignable

print(f"ROOT: {ROOT}")

### Qué hace esta celda

Define el inventario de 14 distritos NASA, la taxonomía agronómica por cultivo, overrides regionales y exclusiones explícitas.

In [ ]:
# ====================================================================
# CELDA 2 — Reglas agronómicas y funciones del generador mapping v2
# ====================================================================
# Propósito:
#   Definir el conocimiento de dominio (distritos NASA, aptitud por cultivo,
#   overrides regionales, exclusiones) y las funciones que asignan cada par
#   (región, cultivo) con producción MIDAGRI > 0 a un distrito climático.
# Metodología v2:
#   1) Taxonomía: lista ordenada de pisos ecológicos compatibles por cultivo.
#   2) Inventario: qué distrito representa cada piso en cada región (14 puntos).
#   3) Override manual cuando la agronomía regional difiere del primer piso.
#   4) Exclusión explícita si no hay intersección piso-cultivo-región.
# Salida: funciones listas; mensaje con conteo de reglas cargadas.
# ====================================================================

# --- Inventario de los 14 distritos NASA POWER (mismo catálogo que notebook 02) ---
# Cada entrada vincula región MIDAGRI, nombre de distrito, y piso ecológico.
# El piso es la clave para cruzar clima NASA con producción agrícola.
DISTRITOS = [
    {"region": "Ica", "distrito": "Chincha Alta", "piso": "costa"},
    {"region": "La Libertad", "distrito": "Viru", "piso": "costa"},
    {"region": "La Libertad", "distrito": "Huamachuco", "piso": "sierra"},
    {"region": "La Libertad", "distrito": "Cascas", "piso": "yunga"},
    {"region": "Piura", "distrito": "Tambogrande", "piso": "bosque_seco"},
    {"region": "Piura", "distrito": "Sullana", "piso": "valle_chira"},
    {"region": "Piura", "distrito": "Canchaque", "piso": "sierra"},
    {"region": "San Martin", "distrito": "Moyobamba", "piso": "selva_alto_mayo"},
    {"region": "San Martin", "distrito": "Tocache", "piso": "selva_huallaga"},
    {"region": "Junin", "distrito": "Perene", "piso": "selva_alta"},
    {"region": "Junin", "distrito": "Rio Tambo", "piso": "selva_baja"},
    {"region": "Junin", "distrito": "El Tambo", "piso": "sierra"},
    {"region": "Puno", "distrito": "Ilave", "piso": "altiplano_lacustre"},
    {"region": "Puno", "distrito": "Ayaviri", "piso": "puna_alta"},
]

# --- Taxonomía agronómica: pisos compatibles por cultivo (orden = prioridad) ---
# El primer piso de la lista es el más representativo agronómicamente.
# Si ese piso existe en la región, se asigna con confianza "alta"; si es el 2.º o 3.º, "media".
# La categoría agronómica agrupa cultivos similares para el reporte documental.
CULTIVO_APTITUD: dict[str, dict] = {
    "esparrago": {"pisos": ["costa"], "categoria": "agroexportacion_costa"},
    "uva": {"pisos": ["yunga", "costa"], "categoria": "frutales"},
    "palta": {"pisos": ["costa", "yunga", "selva_alta", "selva_alto_mayo", "selva_huallaga", "valle_chira"], "categoria": "frutales"},
    "mandarina": {"pisos": ["costa", "yunga", "selva_alta", "selva_alto_mayo", "bosque_seco", "valle_chira", "altiplano_lacustre"], "categoria": "citricos"},
    "naranja": {"pisos": ["costa", "yunga", "selva_alta", "selva_alto_mayo", "selva_huallaga", "bosque_seco", "valle_chira", "sierra", "altiplano_lacustre"], "categoria": "citricos"},
    "limon_sutil": {"pisos": ["costa", "bosque_seco", "yunga", "selva_alta", "selva_alto_mayo"], "categoria": "citricos"},
    "tangelo": {"pisos": ["costa", "selva_alta", "selva_alto_mayo"], "categoria": "citricos"},
    "pimiento": {"pisos": ["costa", "valle_chira", "bosque_seco"], "categoria": "hortalizas_exportacion"},
    "piquillo": {"pisos": ["costa", "valle_chira"], "categoria": "hortalizas_exportacion"},
    "paprika": {"pisos": ["costa", "valle_chira"], "categoria": "hortalizas_exportacion"},
    "tomate": {"pisos": ["costa", "yunga", "sierra", "valle_chira", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "hortalizas"},
    "alcachofa": {"pisos": ["costa", "sierra", "altiplano_lacustre"], "categoria": "hortalizas"},
    "cebolla": {"pisos": ["costa", "sierra", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "aji": {"pisos": ["costa", "valle_chira", "yunga", "selva_alta", "selva_alto_mayo", "sierra"], "categoria": "hortalizas"},
    "ajo": {"pisos": ["costa", "sierra", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "zanahoria": {"pisos": ["costa", "sierra", "yunga", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "zapallo": {"pisos": ["costa", "sierra", "yunga", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "hortalizas"},
    "arandano": {"pisos": ["costa", "sierra"], "categoria": "frutales_bosque"},
    "mango": {"pisos": ["costa", "bosque_seco", "valle_chira", "selva_alta", "selva_alto_mayo", "sierra"], "categoria": "frutales_tropicales"},
    "papaya": {"pisos": ["costa", "bosque_seco", "valle_chira", "selva_alta", "selva_alto_mayo", "sierra", "altiplano_lacustre"], "categoria": "frutales_tropicales"},
    "platano": {"pisos": ["valle_chira", "bosque_seco", "yunga", "selva_alta", "selva_baja", "selva_huallaga", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "frutales_tropicales"},
    "camote": {"pisos": ["costa", "yunga", "sierra", "valle_chira", "selva_alta", "selva_baja", "selva_huallaga", "altiplano_lacustre"], "categoria": "raices"},
    "yuca": {"pisos": ["costa", "valle_chira", "selva_alta", "selva_baja", "selva_huallaga", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "raices"},
    "maiz_amarillo_duro": {"pisos": ["costa", "valle_chira", "yunga", "selva_alta", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "granos"},
    "maiz_chala": {"pisos": ["costa", "valle_chira", "sierra", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "granos"},
    "maiz_choclo": {"pisos": ["costa", "sierra", "valle_chira", "yunga", "altiplano_lacustre"], "categoria": "granos"},
    "maiz_amilaceo": {"pisos": ["sierra", "yunga", "altiplano_lacustre", "valle_chira"], "categoria": "granos_andinos"},
    "arroz_cascara": {"pisos": ["costa", "valle_chira", "selva_alta", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "granos"},
    "cana_para_azucar": {"pisos": ["costa", "valle_chira"], "categoria": "industrial"},
    "algodon_rama": {"pisos": ["costa", "valle_chira", "selva_alto_mayo"], "categoria": "industrial"},
    "aceituna": {"pisos": ["costa", "valle_chira"], "categoria": "frutales"},
    "pallar_grano_seco": {"pisos": ["costa"], "categoria": "leguminosas"},
    "papa": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre", "yunga"], "categoria": "andinos"},
    "quinua": {"pisos": ["puna_alta", "altiplano_lacustre", "sierra"], "categoria": "andinos"},
    "oca": {"pisos": ["puna_alta", "altiplano_lacustre", "sierra"], "categoria": "andinos"},
    "olluco": {"pisos": ["puna_alta", "altiplano_lacustre", "sierra"], "categoria": "andinos"},
    "trigo": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "granos_andinos"},
    "cebada_grano": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "granos_andinos"},
    "cebada_forrajera": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "forrajeros"},
    "avena_forrajera": {"pisos": ["sierra", "puna_alta", "altiplano_lacustre"], "categoria": "forrajeros"},
    "alfalfa": {"pisos": ["costa", "sierra", "puna_alta", "altiplano_lacustre", "valle_chira", "selva_alto_mayo"], "categoria": "forrajeros"},
    "haba_grano_seco": {"pisos": ["costa", "sierra", "altiplano_lacustre"], "categoria": "leguminosas"},
    "frijol_grano_seco": {"pisos": ["costa", "yunga", "sierra", "valle_chira", "altiplano_lacustre", "selva_alto_mayo"], "categoria": "leguminosas"},
    "arveja_grano_seco": {"pisos": ["sierra", "yunga", "altiplano_lacustre", "costa"], "categoria": "leguminosas"},
    "arveja_verde": {"pisos": ["sierra", "yunga", "altiplano_lacustre", "costa"], "categoria": "leguminosas"},
    "manzana": {"pisos": ["yunga", "sierra"], "categoria": "frutales_templados"},
    "melocoton": {"pisos": ["yunga", "sierra"], "categoria": "frutales_templados"},
    "granadilla": {"pisos": ["yunga", "selva_alta", "selva_alto_mayo", "altiplano_lacustre"], "categoria": "frutales"},
    "tuna": {"pisos": ["costa", "sierra", "yunga", "altiplano_lacustre"], "categoria": "frutales"},
    "oregano": {"pisos": ["sierra", "altiplano_lacustre"], "categoria": "aromaticas"},
    "cafe_pergamino": {"pisos": ["selva_alta", "selva_alto_mayo", "sierra", "altiplano_lacustre"], "categoria": "permanentes_selva"},
    "cacao": {"pisos": ["selva_baja", "selva_huallaga", "selva_alta", "selva_alto_mayo"], "categoria": "permanentes_selva"},
    "pina": {"pisos": ["selva_alta", "selva_alto_mayo", "costa", "altiplano_lacustre"], "categoria": "frutales_tropicales"},
    "palma_aceitera": {"pisos": ["selva_baja", "selva_huallaga"], "categoria": "palma"},
}

# --- Overrides regionales documentados (casos donde la taxonomía automática falla) ---
# Clave: (región, cultivo). Valor: (piso, distrito, nivel_confianza, justificación_texto).
# Ejemplo: uva en La Libertad va a Cascas (yunga), no a Virú (costa).
OVERRIDES: dict[tuple[str, str], tuple[str, str, str, str]] = {
    ("La Libertad", "uva"): ("yunga", "Cascas", "alta", "Uva de mesa en ceja de selva/yunga (Cascas), no en costa virú."),
    ("La Libertad", "cana_para_azucar"): ("costa", "Viru", "alta", "Agroindustria azucarera concentrada en costa norte (Virú)."),
    ("La Libertad", "frijol_grano_seco"): ("yunga", "Cascas", "alta", "Frijol en ceja de selva/yunga (Cascas), coherente con v1 revisado."),
    ("La Libertad", "haba_grano_seco"): ("sierra", "Huamachuco", "alta", "Haba en sierra libertense (Huamachuco)."),
    ("La Libertad", "pina"): ("yunga", "Cascas", "media", "Piña en yunga; costa virú menos representativa."),
    ("Junin", "cacao"): ("selva_baja", "Rio Tambo", "alta", "Cacao en selva baja de la región (Corrección documentada del grupo)."),
    ("Junin", "yuca"): ("selva_baja", "Rio Tambo", "alta", "Yuca y cacao comparten selva baja amazónica."),
    ("Junin", "tangelo"): ("selva_alta", "Perene", "alta", "Cítricos en ceja de selva (Perené)."),
    ("San Martin", "cacao"): ("selva_huallaga", "Tocache", "alta", "Corredor cacao coca histórico en Tocache (selva huallaga)."),
    ("San Martin", "platano"): ("selva_huallaga", "Tocache", "alta", "Plátano en selva baja/huallaga."),
    ("Piura", "uva"): ("valle_chira", "Sullana", "media", "Uva de mesa en valle de Chira/Sullana; producción regional significativa."),
    ("Piura", "platano"): ("valle_chira", "Sullana", "alta", "Plátano dominante en valle de Chira."),
    ("Piura", "limon_sutil"): ("bosque_seco", "Tambogrande", "alta", "Limón en bosque seco piurano."),
    ("Piura", "mango"): ("bosque_seco", "Tambogrande", "alta", "Mango en bosque seco."),
    ("Puno", "quinua"): ("puna_alta", "Ayaviri", "alta", "Quinua en puna alta (Ayaviri)."),
    ("Puno", "papa"): ("puna_alta", "Ayaviri", "alta", "Papa en puna; mayor volumen en Ayaviri vs altiplano lacustre."),
    ("Puno", "alfalfa"): ("altiplano_lacustre", "Ilave", "alta", "Alfalfa forrajera en altiplano lacustre (Ilave)."),
    ("Puno", "avena_forrajera"): ("puna_alta", "Ayaviri", "alta", "Avena forrajera en puna alta."),
    ("Ica", "papa"): ("costa", "Chincha Alta", "baja", "Producción MIDAGRI regional incluye sierra alta de Ica; único distrito NASA es costa → proxy débil."),
    ("Ica", "maiz_choclo"): ("costa", "Chincha Alta", "media", "Maíz choclo en valle de Ica costa; parte puede provenir de zonas altas."),
}

# --- Exclusiones explícitas: combos que NO deben mapearse aunque tengan producción MIDAGRI ---
# Motivo documentado por fila (volumen marginal, incompatibilidad agronómica, etc.).
EXCLUIR_EXPLICITO: dict[tuple[str, str], str] = {
    ("Ica", "quinua"): "Volumen marginal (<200 t); sin piso andino en distritos Ica.",
    ("Ica", "oca"): "Sin aptitud costera; volumen residual.",
    ("Ica", "olluco"): "Sin aptitud costera; volumen residual.",
    ("Puno", "cacao"): "Cacao no es cultivo de altiplano; probable registro residual.",
    ("La Libertad", "cacao"): "Sin volumen agronómico relevante en costa/sierra libertense.",
}


def construir_inventario_region() -> dict[str, dict[str, str]]:
    """Construye mapa región → {piso: distrito} a partir de la lista DISTRITOS."""
    inv: dict[str, dict[str, str]] = {}
    for d in DISTRITOS:
        inv.setdefault(d["region"], {})[d["piso"]] = d["distrito"]
    return inv


def cargar_produccion(ruta_midagri: Path) -> pd.DataFrame:
    """Lee midagri_largo.csv y devuelve solo pares (región, cultivo) con producción total > 0."""
    mid = pd.read_csv(ruta_midagri)
    # sum(min_count=1): si todos los meses son NaN, el total queda NaN (no 0 artificial)
    prod = (
        mid.groupby(["region", "cultivo"], as_index=False)["produccion_mensual"]
        .sum(min_count=1)
        .rename(columns={"produccion_mensual": "produccion_total_ton"})
    )
    return prod[prod["produccion_total_ton"] > 0].copy()


def asignar(
    region: str,
    cultivo: str,
    produccion: float,
    inventario: dict[str, dict[str, str]],
) -> dict | None:
    """
    Asigna un distrito a un par (región, cultivo) siguiendo la prioridad:
    1) Exclusión explícita → None
    2) Override regional documentado → fila con regla_asignacion='override_regional'
    3) Taxonomía: primer piso compatible en la región → 'taxonomia_piso_prioridad'
    4) Sin piso compatible → None (se registrará en excluidos)
    """
    key = (region, cultivo)

    if key in EXCLUIR_EXPLICITO:
        return None

    if key in OVERRIDES:
        piso, distrito, conf, just = OVERRIDES[key]
        return {
            "region": region,
            "cultivo": cultivo,
            "piso_asignado": piso,
            "distrito": distrito,
            "produccion_total_ton": round(produccion, 2),
            "nivel_confianza": conf,
            "regla_asignacion": "override_regional",
            "categoria_agronomica": CULTIVO_APTITUD.get(cultivo, {}).get("categoria", "sin_clasificar"),
            "justificacion": just,
        }

    apt = CULTIVO_APTITUD.get(cultivo)
    if apt is None:
        return None

    pisos_region = inventario.get(region, {})
    # Recorrer pisos en orden de prioridad agronómica hasta encontrar uno presente en la región
    for piso in apt["pisos"]:
        if piso in pisos_region:
            conf = "alta" if piso == apt["pisos"][0] else "media"
            return {
                "region": region,
                "cultivo": cultivo,
                "piso_asignado": piso,
                "distrito": pisos_region[piso],
                "produccion_total_ton": round(produccion, 2),
                "nivel_confianza": conf,
                "regla_asignacion": "taxonomia_piso_prioridad",
                "categoria_agronomica": apt["categoria"],
                "justificacion": (
                    f"Primer piso compatible en {region}: {piso} "
                    f"(prioridad agronómica {apt['pisos'].index(piso) + 1}/{len(apt['pisos'])})."
                ),
            }

    return None


def generar_mapping_v2(prod: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Itera todos los combos con producción y construye df_map (asignados) y df_exc (excluidos)."""
    inventario = construir_inventario_region()
    asignados: list[dict] = []
    excluidos: list[dict] = []

    for _, row in prod.sort_values(["region", "cultivo"]).iterrows():
        region, cultivo, ton = row["region"], row["cultivo"], row["produccion_total_ton"]
        key = (region, cultivo)

        if key in EXCLUIR_EXPLICITO:
            excluidos.append({
                "region": region,
                "cultivo": cultivo,
                "produccion_total_ton": round(ton, 2),
                "motivo_exclusion": EXCLUIR_EXPLICITO[key],
            })
            continue

        res = asignar(region, cultivo, ton, inventario)
        if res:
            asignados.append(res)
        else:
            pisos_disp = list(inventario.get(region, {}).keys())
            apt_pisos = CULTIVO_APTITUD.get(cultivo, {}).get("pisos", [])
            excluidos.append({
                "region": region,
                "cultivo": cultivo,
                "produccion_total_ton": round(ton, 2),
                "motivo_exclusion": (
                    f"Sin intersección entre pisos del cultivo {apt_pisos} "
                    f"y pisos disponibles en {region} {pisos_disp}."
                ),
            })

    df_map = pd.DataFrame(asignados).sort_values(["region", "cultivo"]).reset_index(drop=True)
    df_exc = pd.DataFrame(excluidos).sort_values(["region", "cultivo"]).reset_index(drop=True)
    return df_map, df_exc


def comparar_v1_v2(v1: pd.DataFrame, v2: pd.DataFrame, prod: pd.DataFrame) -> pd.DataFrame:
    """Cruza mapping v1 y v2 con la producción para etiquetar cambios de distrito."""
    cols_key = ["region", "cultivo"]
    m1 = v1[cols_key + ["piso_asignado", "distrito"]].rename(
        columns={"piso_asignado": "piso_v1", "distrito": "distrito_v1"}
    )
    m2 = v2[cols_key + ["piso_asignado", "distrito", "nivel_confianza", "justificacion"]].rename(
        columns={"piso_asignado": "piso_v2", "distrito": "distrito_v2"}
    )

    comp = prod.merge(m1, on=cols_key, how="left")
    comp = comp.merge(m2, on=cols_key, how="left")

    # Clasificar cada fila según si el distrito cambió, es nuevo, solo en v1, o igual
    comp["estado"] = "sin_cambio"
    mask_new = comp["distrito_v1"].isna() & comp["distrito_v2"].notna()
    mask_gone = comp["distrito_v1"].notna() & comp["distrito_v2"].isna()
    mask_diff = (
        comp["distrito_v1"].notna()
        & comp["distrito_v2"].notna()
        & (comp["distrito_v1"] != comp["distrito_v2"])
    )
    comp.loc[mask_new, "estado"] = "nuevo_en_v2"
    comp.loc[mask_gone, "estado"] = "solo_en_v1"
    comp.loc[mask_diff, "estado"] = "distrito_cambiado"
    comp.loc[
        comp["distrito_v1"].notna()
        & comp["distrito_v2"].notna()
        & (comp["distrito_v1"] == comp["distrito_v2"]),
        "estado",
    ] = "sin_cambio"

    return comp.sort_values(["estado", "region", "cultivo"])


def exportar_mapping_pipeline(df: pd.DataFrame, ruta: Path) -> None:
    """Versión reducida de 4 columnas que consume el notebook 03 en el merge."""
    out = df[["region", "cultivo", "piso_asignado", "distrito"]].copy()
    out.to_csv(ruta, index=False, encoding="utf-8-sig")


def escribir_reporte_md(
    v1: pd.DataFrame,
    v2: pd.DataFrame,
    excl: pd.DataFrame,
    comp: pd.DataFrame,
    ruta: Path,
) -> None:
    """Genera reporte markdown comparativo v1 vs v2 para DOCUMENTACIÓN/."""
    n_prod = len(comp)
    n_v1 = len(v1)
    n_v2 = len(v2)
    cambios = comp[comp["estado"] == "distrito_cambiado"]
    nuevos = comp[comp["estado"] == "nuevo_en_v2"]
    solo_v1 = comp[comp["estado"] == "solo_en_v1"]

    ton_cubierta = comp.loc[comp["distrito_v2"].notna(), "produccion_total_ton"].sum()
    ton_total = comp["produccion_total_ton"].sum()

    lines = [
        "# Reporte comparativo: mapping v1 vs v2",
        "",
        f"**Generado por:** `SCRIPTS/notebooks/00_build_mapping_cultivo_distrito.ipynb`",
        "",
        "## Resumen ejecutivo",
        "",
        "| Métrica | v1 (legacy) | v2 (taxonomía) |",
        "|---------|------------:|---------------:|",
        f"| Filas mapping | {n_v1} | {n_v2} |",
        f"| Combos con producción MIDAGRI | {n_prod} | {n_prod} |",
        f"| Distrito cambiado | — | {len(cambios)} |",
        f"| Nuevos mapeos | — | {len(nuevos)} |",
        f"| Solo en v1 (sin prod. o reemplazados) | — | {len(solo_v1)} |",
        f"| Excluidos explícitos v2 | — | {len(excl)} |",
        f"| Producción cubierta por v2 | — | {ton_cubierta / ton_total * 100:.1f}% ({ton_cubierta:,.0f} t) |",
        "",
        "## Mejoras metodológicas v2",
        "",
        "1. **Taxonomía agronómica** por cultivo (pisos compatibles ordenados).",
        "2. **Solo combina con producción real** MIDAGRI > 0.",
        "3. **Overrides regionales documentados** (uva Libertad→Cascas, cacao→Tocache, etc.).",
        "4. **Nivel de confianza** (alta/media/baja) y justificación por fila.",
        "5. **Exclusión explícita** cuando cultivo y pisos regionales son incompatibles.",
        "",
        "## Cambios de distrito (v1 → v2)",
        "",
    ]

    if len(cambios) == 0:
        lines.append("_No hay cambios de distrito._")
    else:
        lines.append("| Región | Cultivo | v1 | v2 | Confianza |")
        lines.append("|--------|---------|----|----|-----------|")
        for _, r in cambios.iterrows():
            lines.append(
                f"| {r['region']} | {r['cultivo']} | {r['distrito_v1']} | {r['distrito_v2']} | {r.get('nivel_confianza', '')} |"
            )

    lines.extend(["", "## Nuevos mapeos en v2 (con producción, ausentes en v1)", ""])
    if len(nuevos) == 0:
        lines.append("_Ninguno._")
    else:
        lines.append("| Región | Cultivo | Distrito | Prod. (t) | Confianza |")
        lines.append("|--------|---------|----------|----------:|-----------|")
        for _, r in nuevos.head(40).iterrows():
            lines.append(
                f"| {r['region']} | {r['cultivo']} | {r['distrito_v2']} | {r['produccion_total_ton']:,.0f} | {r.get('nivel_confianza', '')} |"
            )
        if len(nuevos) > 40:
            lines.append(f"\n_... y {len(nuevos) - 40} más (ver CSV)._")

    lines.extend(["", "## Exclusiones v2", ""])
    if len(excl) == 0:
        lines.append("_Ninguna._")
    else:
        for _, r in excl.iterrows():
            lines.append(
                f"- **{r['region']} / {r['cultivo']}** ({r['produccion_total_ton']:,.0f} t): {r['motivo_exclusion']}"
            )

    lines.extend([
        "",
        "## Uso en el pipeline",
        "",
        "El notebook 03 consume `BDS/mapping/mapping_cultivo_distrito_v2_pipeline.csv`.",
        "",
        "Regenerar mapping:",
        "",
        "```bash",
        "jupyter execute SCRIPTS/notebooks/00_build_mapping_cultivo_distrito.ipynb",
        "```",
    ])

    ruta.write_text("\n".join(lines), encoding="utf-8")

print(f"Reglas cargadas: {len(CULTIVO_APTITUD)} cultivos, {len(OVERRIDES)} overrides, {len(DISTRITOS)} distritos")


### Qué hace esta celda

Ejecuta el generador, escribe los CSV en `BDS/mapping/` y muestra un resumen.

In [ ]:
# ====================================================================
# CELDA 3 — Ejecución: generar mapping v2 y exportar archivos
# ====================================================================
# Propósito:
#   Validar insumos (midagri_largo.csv y mapping v1), ejecutar el generador v2,
#   comparar con v1, y escribir todos los CSV y el reporte markdown.
# Archivos generados:
#   - BDS/mapping/mapping_cultivo_distrito_v2.csv (completo con metadatos)
#   - BDS/mapping/mapping_cultivo_distrito_v2_pipeline.csv (4 cols para notebook 03)
#   - OUTPUTS/mapping_v2_excluidos.csv, reporte_mapping_v1_vs_v2.csv
#   - DOCUMENTACIÓN/reporte_mapping_v1_vs_v2.md
# ====================================================================

# El notebook 01 debe haber corrido antes: sin producción MIDAGRI no hay mapping
if not RUTA_MIDAGRI.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_MIDAGRI}. Ejecuta primero 01_midagri_pipeline.ipynb."
    )
# Mapping v1 es referencia obligatoria para la comparación v1 vs v2
if not RUTA_V1.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_V1}. Se requiere el mapping v1 de referencia para la comparación."
    )

# Cargar producción agregada (solo combos con toneladas > 0)
prod = cargar_produccion(RUTA_MIDAGRI)
v1 = pd.read_csv(RUTA_V1)

# Respaldo automático de v1 la primera vez (no sobrescribe si ya existe legacy)
if not RUTA_V1_LEGACY.exists():
    v1.to_csv(RUTA_V1_LEGACY, index=False, encoding="utf-8-sig")
    print(f"  [OK] respaldo v1: {RUTA_V1_LEGACY}")

# Generar mapping v2 y tabla de excluidos
v2, excl = generar_mapping_v2(prod)
# Cruzar v1, v2 y producción para auditar cambios
comp = comparar_v1_v2(v1, v2, prod)

# --- Exportación de todos los artefactos ---
v2.to_csv(RUTA_V2, index=False, encoding="utf-8-sig")
exportar_mapping_pipeline(v2, RUTA_V2_PIPELINE)
excl.to_csv(RUTA_EXCL, index=False, encoding="utf-8-sig")
comp.to_csv(RUTA_COMP, index=False, encoding="utf-8-sig")
escribir_reporte_md(v1, v2, excl, comp, RUTA_MD)

# --- Resumen en consola para verificación rápida ---
print("=== Mapping v2 generado ===")
print(f"  Filas v1: {len(v1)}")
print(f"  Filas v2: {len(v2)}")
print(f"  Excluidos: {len(excl)}")
print(f"  Cambios distrito: {(comp['estado'] == 'distrito_cambiado').sum()}")
print(f"  Nuevos en v2: {(comp['estado'] == 'nuevo_en_v2').sum()}")
ton_cov = comp.loc[comp["distrito_v2"].notna(), "produccion_total_ton"].sum()
print(f"  Producción cubierta v2: {100 * ton_cov / comp['produccion_total_ton'].sum():.1f}%")
print(f"\n  {RUTA_V2}")
print(f"  {RUTA_V2_PIPELINE}")
print(f"  {RUTA_MD}")

print(v2.head(10).to_string())
print("\nConfianza:")
print(v2["nivel_confianza"].value_counts().to_string())
print("\nReglas:")
print(v2["regla_asignacion"].value_counts().to_string())